# Experiment 13: Risk Scoring

This experiment converts model probabilities into interpretable risk scores and risk categories to support maintenance decision-making.

In [2]:
from pathlib import Path
import sys

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data.load_data import (
    load_ai4i_dataset
)

from src.features.feature_engineering import (
    create_engineered_features
)

from src.features.preprocessing import (
    prepare_modeling_dataset
)

from src.models.champion_model import (
    build_champion_xgboost_model
)

from src.scoring.risk_scoring import (
    calculate_risk_score,
    classify_risk_level
)

from sklearn.model_selection import (
    train_test_split
)

# Data

df = load_ai4i_dataset()

df = create_engineered_features(df)

df = prepare_modeling_dataset(df)

X = df.drop(
    columns=["Machine failure"]
)

y = df["Machine failure"]

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y
)

# Train champion model

model = build_champion_xgboost_model()

model.fit(
    X_train,
    y_train
)

# Example machine

sample = X_test.iloc[[0]]

probability = model.predict_proba(
    sample
)[0][1]

risk_score = calculate_risk_score(
    probability
)

risk_level = classify_risk_level(
    probability
)

print(
    f"Failure Probability: {probability:.4f}"
)

print(
    f"Risk Score: {risk_score}"
)

print(
    f"Risk Level: {risk_level}"
)

Failure Probability: 0.0242
Risk Score: 2.42
Risk Level: LOW


## Prediction Report Generation

This experiment combines model probability, risk score, and risk category into a unified prediction report suitable for deployment.

In [3]:
from src.scoring.prediction_report import (
    create_prediction_report
)

sample = X_test.iloc[[0]]

failure_probability = model.predict_proba(
    sample
)[0][1]

report = create_prediction_report(
    failure_probability
)

report

{'failure_probability': 0.0242, 'risk_score': 2.42, 'risk_level': 'LOW'}